# Inspect the off-shell HZZ ntuple

**Input:** `My Drive/Colab Notebooks/HZZ_OffShell/dataset_wDR_mean_origIndex.h5`

In Colab, choose **Runtime → Run all**, then authorize the Drive mount. A CPU runtime is sufficient. The notebook opens the original HDF5 file read-only and reads small samples without copying the full file into RAM.

It reports groups, dataset shapes, data types, stored attributes, available column labels, example entries, and sampled numerical/categorical diagnostics. Negative weights are retained and counted. Dataset shapes are exact; sampled counts and distributions are not full-file totals.

The last cell writes `.inspection.txt` and `.inspection.json` reports beside the input file in Drive. After running, tell me the reports are ready; I can read them directly. The reports contain a few example entries, so keep them in Drive rather than committing them to the public repository.

## 1. Mount Drive and set the input

The default path matches the project folder. To inspect a different file or run locally, change `FILE_PATH`. `SAMPLE_ROWS` is a maximum per dataset, split across up to five small windows; it is not a random sample.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import json
import math
import pickletools

import h5py
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from google.colab import drive
except ImportError:
    print("Local runtime: set FILE_PATH to your local HDF5 file below.")
else:
    drive.mount("/content/drive")

print(f"h5py {h5py.__version__} | NumPy {np.__version__} | pandas {pd.__version__}")

Mounted at /content/drive
h5py 3.14.0 | NumPy 2.1.3 | pandas 2.2.3


In [2]:
FILE_PATH = Path("/content/drive/MyDrive/Colab Notebooks/HZZ_OffShell/dataset_wDR_mean_origIndex.h5")

SAMPLE_ROWS = 2000
PREVIEW_ROWS = 3
MAX_SAMPLE_BYTES = 8 * 1024**2  # Fixed-size requested data per dataset; compression can require extra I/O.
MAX_DIAGNOSTIC_COLUMNS = 512

assert SAMPLE_ROWS > 0 and PREVIEW_ROWS > 0 and MAX_SAMPLE_BYTES > 0
if not FILE_PATH.is_file():
    raise FileNotFoundError(f"File not found: {FILE_PATH}\nCheck the Drive mount and FILE_PATH.")
print(f"Input: {FILE_PATH}")
print(f"File size: {FILE_PATH.stat().st_size:,} bytes ({FILE_PATH.stat().st_size / 1024**3:.3f} GiB)")

Input: /content/drive/MyDrive/Colab Notebooks/HZZ_OffShell/dataset_wDR_mean_origIndex.h5
File size: 3,626,490,080 bytes (3.377 GiB)


## 2. Inspection helpers

These helpers use `h5py` for structural inspection. They recognize compound fields and common column-label conventions, including pandas fixed-format blocks. Variable-length non-text object payloads are left unread; metadata is never unpickled. Any sampling error is recorded with its dataset path.

In [3]:
def json_value(value, limit=10000):
    """Make HDF5/NumPy values readable without deserializing pickled objects."""
    if isinstance(value, np.void) and value.dtype.names:
        return {name: json_value(value[name], limit) for name in value.dtype.names}
    if isinstance(value, np.ndarray):
        if value.size > limit:
            return {"shape": list(value.shape), "first_values": json_value(value.ravel()[:limit]),
                    "truncated": True}
        return json_value(value.tolist(), limit)
    if isinstance(value, np.generic):
        return json_value(value.item(), limit)
    if isinstance(value, bytes):
        return value.decode("utf-8", errors="backslashreplace")
    if isinstance(value, (list, tuple)):
        return [json_value(x, limit) for x in value]
    if isinstance(value, dict):
        return {str(k): json_value(v, limit) for k, v in value.items()}
    if isinstance(value, float) and not np.isfinite(value):
        return str(value)
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    return repr(value)


def read_attributes(obj):
    result = {}
    for name in obj.attrs:
        try:
            aid = obj.attrs.get_id(name)
            if aid.shape is None:
                result[name] = {"null_attribute": True, "dtype": str(aid.dtype)}
            elif math.prod(aid.shape) * aid.dtype.itemsize > MAX_SAMPLE_BYTES:
                result[name] = {"skipped_large_attribute": True, "shape": list(aid.shape)}
            else:
                result[name] = json_value(obj.attrs[name])
        except Exception as exc:
            result[name] = {"read_error": str(exc)}
    return result


def read_sample(ds):
    """Read up to five small, nonoverlapping windows spread across the row axis."""
    if ds.shape is None or ds.size == 0:
        return None, [], "Null or empty dataset."
    is_text = h5py.check_string_dtype(ds.dtype) is not None
    if ds.dtype.hasobject and not is_text:
        return None, [], "Variable-length/object payload: schema only; no unpickling."
    if ds.is_virtual or ds.external:
        return None, [], "External/virtual data: schema only."
    row_bytes = math.prod(ds.shape[1:]) * max(ds.dtype.itemsize, 1) if ds.ndim else ds.dtype.itemsize
    if row_bytes > MAX_SAMPLE_BYTES:
        return None, [], "A single row exceeds the sample byte budget."
    if ds.ndim == 0:
        return np.asarray(ds[()]), ["scalar"], None
    target = min(SAMPLE_ROWS, ds.shape[0], max(1, MAX_SAMPLE_BYTES // max(1, row_bytes)))
    # Variable-length strings do not have a known byte size; keep their sample tiny.
    if ds.dtype.hasobject:
        target = min(target, 16)
    if target >= ds.shape[0]:
        return ds[:], [[0, ds.shape[0]]], None
    blocks = min(5, target)
    width = max(1, target // blocks)
    starts = np.linspace(0, ds.shape[0] - width, blocks, dtype=np.int64)
    slices = [[int(start), int(start + width)] for start in starts]
    return np.concatenate([ds[a:b] for a, b in slices], axis=0), slices, None


def string_list(value):
    """Read plain labels or pandas' simple protocol-0 string list, without unpickling."""
    raw = json_value(value)
    if isinstance(raw, list) and all(isinstance(x, str) for x in raw):
        return raw
    if not isinstance(raw, str) or len(raw) > 100000:
        return None
    try:
        ops = list(pickletools.genops(raw.encode("utf-8")))
        names = [op.name for op, _, _ in ops]
        if names[:3] != ["MARK", "LIST", "PUT"] or names[-1:] != ["STOP"]:
            return None
        middle = ops[3:-1]
        if len(middle) % 3:
            return None
        labels = []
        for i in range(0, len(middle), 3):
            if [op.name for op, _, _ in middle[i:i+3]] != ["UNICODE", "PUT", "APPEND"]:
                return None
            labels.append(middle[i][1])
        return labels
    except Exception:
        return None


def find_column_labels(ds):
    """Recover common h5py and pandas-fixed column label conventions."""
    expected = ds.shape[1] if ds.shape is not None and ds.ndim == 2 else None
    if expected is None:
        return None
    candidates = ["columns", "column_names", "feature_names", "variables"]
    for key in candidates:
        if key in ds.attrs:
            raw = string_list(ds.attrs[key])
            if raw is not None and len(raw) == expected:
                return raw
    leaf = ds.name.rsplit("/", 1)[-1]
    if leaf.endswith("_values"):
        candidates.insert(0, leaf[:-7] + "_items")
    candidates.append("axis0")
    for key in candidates:
        # Do not follow soft links or external files while guessing label locations.
        if not isinstance(ds.parent.get(key, getlink=True), h5py.HardLink):
            continue
        obj = ds.parent[key]
        if (isinstance(obj, h5py.Dataset) and obj.shape == (expected,)
                and expected <= 10000 and h5py.check_string_dtype(obj.dtype) is not None):
            return json_value(obj[:])
    return None


def describe_values(values):
    arr = np.asarray(values).ravel()
    result = {"sampled_values": int(arr.size), "dtype": str(arr.dtype)}
    if arr.dtype.kind in "biuf":
        finite_mask = np.isfinite(arr)
        finite = arr[finite_mask]
        result.update({"nonfinite": int((~finite_mask).sum()),
                       "nan": int(np.isnan(arr).sum()),
                       "positive_infinity": int(np.isposinf(arr).sum()),
                       "negative_infinity": int(np.isneginf(arr).sum())})
        if finite.size:
            result.update({"min": json_value(finite.min()), "max": json_value(finite.max()),
                           "mean": json_value(float(np.mean(finite, dtype=np.float64))),
                           "std": json_value(float(np.std(finite, dtype=np.float64))),
                           "negative": int((finite < 0).sum()), "zero": int((finite == 0).sum()),
                           "positive": int((finite > 0).sum())})
            unique, counts = np.unique(finite, return_counts=True)
            if len(unique) <= 20:
                result["sample_value_counts"] = [{"value": json_value(v), "count": int(n)}
                                                 for v, n in zip(unique, counts)]
    elif arr.dtype.kind in "SUO":
        labels = [str(json_value(x)) for x in arr]
        counts = Counter(labels)
        result["unique_in_sample"] = len(counts)
        result["most_common_in_sample"] = counts.most_common(20)
    return result


def describe_sample(sample, labels=None, field_labels=None):
    result = {}
    field_labels = field_labels or {}

    def add_array(name, arr, labels_for_array=None):
        arr = np.asarray(arr)
        if arr.dtype.names:
            for field in arr.dtype.names:
                add_array(f"{name}.{field}" if name else field, arr[field], field_labels.get(field))
            return
        if arr.ndim <= 1:
            result[name or "value"] = describe_values(arr)
            return
        width = math.prod(arr.shape[1:])
        if width > MAX_DIAGNOSTIC_COLUMNS:
            result[name or "all_components"] = describe_values(arr)
            result[name or "all_components"]["note"] = "Components pooled; see shape and labels."
            return
        flat = arr.reshape(arr.shape[0], width)
        for index in range(width):
            key = str(labels_for_array[index]) if labels_for_array else f"{name or 'column'}[{index}]"
            if key in result:
                key = f"{key} [component {index}]"
            result[key] = describe_values(flat[:, index])

    add_array("", sample, labels)
    return result


def inspect_hdf5(path):
    stat = path.stat()
    report = {
        "file": str(path), "size_bytes": stat.st_size,
        "inspected_utc": datetime.now(timezone.utc).isoformat(),
        "versions": {"h5py": h5py.__version__, "numpy": np.__version__, "pandas": pd.__version__},
        "settings": {"sample_rows_per_dataset": SAMPLE_ROWS, "sample_bytes_per_dataset": MAX_SAMPLE_BYTES},
        "notes": [
            "Dataset shapes are exact metadata. Axis 0 is not automatically an event count.",
            "Statistics use at most five small contiguous windows spread across each dataset.",
            "Samples are deterministic diagnostics, not random or guaranteed representative estimates.",
            "Negative/nonfinite counts and categorical counts apply only to the sampled values.",
            "Pandas fixed-format labels are recovered where available; table labels also appear in attributes.",
            "No units, process meanings, weight normalization, or full-file validity are inferred.",
            "Large compressed chunks may require extra I/O despite small requested slices.",
        ],
        "groups": [], "datasets": [], "errors": [],
    }
    if not h5py.is_hdf5(path):
        raise ValueError(f"Not a readable HDF5 file: {path}")
    with h5py.File(path, "r") as handle:
        report["root_attributes"] = read_attributes(handle)

        def visit(name, obj):
            if isinstance(obj, h5py.Group):
                report["groups"].append({"path": obj.name, "attributes": read_attributes(obj)})
                return
            item = {"path": obj.name, "shape": list(obj.shape) if obj.shape is not None else None,
                    "dtype": str(obj.dtype), "field_names": list(obj.dtype.names or []),
                    "logical_bytes": int(obj.size * obj.dtype.itemsize) if obj.size is not None else 0,
                    "stored_bytes": int(obj.id.get_storage_size()), "chunks": json_value(obj.chunks),
                    "compression": obj.compression, "attributes": read_attributes(obj)}
            report["datasets"].append(item)
            try:
                item["column_labels"] = find_column_labels(obj)
                item["field_column_labels"] = {}
                for field in obj.dtype.names or []:
                    key = field + "_kind"
                    if key in obj.attrs:
                        labels = string_list(obj.attrs[key])
                        if labels is not None and len(labels) == math.prod(obj.dtype[field].shape):
                            item["field_column_labels"][field] = labels
                sample, slices, note = read_sample(obj)
                item["sample_slices_start_stop"] = slices
                if note:
                    item["sample_note"] = note
                if sample is not None:
                    item["sample_shape"] = list(sample.shape)
                    preview = sample if sample.ndim == 0 else sample[:PREVIEW_ROWS]
                    item["preview"] = json_value(preview, limit=300)
                    item["sample_statistics"] = describe_sample(sample, item["column_labels"], item["field_column_labels"])
            except Exception as exc:
                item["sample_error"] = str(exc)
                report["errors"].append({"path": obj.name, "error": str(exc)})

        handle.visititems(visit)
    return report


def make_text_report(report):
    lines = ["HDF5 INSPECTION REPORT", f"File: {report['file']}",
             f"Size: {report['size_bytes']:,} bytes", f"Inspected: {report['inspected_utc']}", ""]
    lines += ["INTERPRETATION"] + ["- " + note for note in report["notes"]]
    lines += ["", "ROOT ATTRIBUTES", json.dumps(report["root_attributes"], indent=2, ensure_ascii=False)]
    for group in report["groups"]:
        lines += ["", "GROUP " + group["path"], json.dumps(group["attributes"], indent=2, ensure_ascii=False)]
    for item in report["datasets"]:
        lines += ["", "DATASET " + item["path"], json.dumps(item, indent=2, ensure_ascii=False)]
    lines += ["", "READ ERRORS", json.dumps(report["errors"], indent=2, ensure_ascii=False)]
    return "\n".join(lines) + "\n"


## 3. Read the schema and bounded samples

This may take a little time on the mounted Drive. HDF5 chunk compression can require reading more bytes than the requested slices. This is an inspection, not a complete integrity scan.

In [4]:
report = inspect_hdf5(FILE_PATH)

schema_rows = []
for item in report["datasets"]:
    schema_rows.append({
        "path": item["path"], "shape": item["shape"], "dtype": item["dtype"],
        "stored_MiB": round(item["stored_bytes"] / 1024**2, 3),
        "compression": item["compression"], "sample_shape": item.get("sample_shape"),
        "note": item.get("sample_error", item.get("sample_note", "")),
    })

print(f"Found {len(report['groups'])} groups and {len(report['datasets'])} datasets.")
print("Shapes are exact; the meaning of each axis must be checked from the layout and labels.")
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 100):
    display(pd.DataFrame(schema_rows))
if report["errors"]:
    print("Some samples could not be read; the report includes these errors:")
    display(pd.DataFrame(report["errors"]))

Found 1 groups and 12 datasets.
Shapes are exact; the meaning of each axis must be checked from the layout and labels.


,path,shape,dtype,stored_MiB,compression,sample_shape,note
0,/dataset/axis0,[66],|S16,0.001,None,[66],
1,/dataset/axis1,[9540480],int64,72.788,None,[2000],
2,/dataset/block0_items,[34],|S16,0.001,None,[34],
3,/dataset/block0_values,"[9540480, 34]",float32,1237.397,None,"[2000, 34]",
4,/dataset/block1_items,[21],|S12,0.000,None,[21],
5,/dataset/block1_values,"[9540480, 21]",float64,1528.550,None,"[2000, 21]",
6,/dataset/block2_items,[2],|S10,0.000,None,[2],
7,/dataset/block2_values,"[9540480, 2]",int32,72.788,None,"[2000, 2]",
8,/dataset/block3_items,[7],|S5,0.000,None,[7],
9,/dataset/block3_values,"[9540480, 7]",int64,509.517,None,"[2000, 7]",


## 4. Attributes, column names, and example entries

This output can reveal feature names, process labels, event identifiers, weight definitions, or preprocessing metadata when they are actually stored in the file. Names alone do not establish units or physics conventions.

In [5]:
print("ROOT ATTRIBUTES")
print(json.dumps(report["root_attributes"], indent=2, ensure_ascii=False))
for group in report["groups"]:
    print("\nGROUP", group["path"])
    print(json.dumps(group["attributes"], indent=2, ensure_ascii=False))

for item in report["datasets"]:
    print("\nDATASET", item["path"], "| shape:", item["shape"], "| dtype:", item["dtype"])
    for key in ("field_names", "column_labels", "field_column_labels", "attributes", "sample_slices_start_stop", "preview", "sample_note", "sample_error"):
        if item.get(key) is not None and item.get(key) != [] and item.get(key) != {}:
            print(key + ":", json.dumps(item[key], indent=2, ensure_ascii=False))

ROOT ATTRIBUTES
{
  "CLASS": "GROUP",
  "PYTABLES_FORMAT_VERSION": "2.1",
  "TITLE": {
    "null_attribute": true,
    "dtype": "|S1"
  },
  "VERSION": "1.0"
}

GROUP /dataset
{
  "CLASS": "GROUP",
  "TITLE": {
    "null_attribute": true,
    "dtype": "|S1"
  },
  "VERSION": "1.0",
  "axis0_variety": "regular",
  "axis1_variety": "regular",
  "block0_items_variety": "regular",
  "block1_items_variety": "regular",
  "block2_items_variety": "regular",
  "block3_items_variety": "regular",
  "block4_items_variety": "regular",
  "encoding": "UTF-8",
  "errors": "strict",
  "nblocks": 5,
  "ndim": 2,
  "pandas_type": "frame",
  "pandas_version": "0.15.2"
}

DATASET /dataset/axis0 | shape: [66] | dtype: |S16
attributes: {
  "CLASS": "ARRAY",
  "FLAVOR": "numpy",
  "TITLE": {
    "null_attribute": true,
    "dtype": "|S1"
  },
  "VERSION": "2.4",
  "kind": "string",
  "name": "N.",
  "transposed": 1
}
sample_slices_start_stop: [
  [
    0,
    66
  ]
]
preview: [
  "MCFM_dxs_HZZ",
  "MCFM_dxs_

## 5. Sample diagnostics

For numerical columns: finite ranges, mean and standard deviation, NaN/infinity counts, and negative/zero/positive counts. Low-cardinality values and string labels also receive sample frequency counts. A zero negative count does not prove the whole dataset has no negative weights; no luminosity or yield normalization is assumed.

In [6]:
for item in report["datasets"]:
    stats = item.get("sample_statistics", {})
    if not stats:
        continue
    print("\nSAMPLE DIAGNOSTICS:", item["path"])
    rows = [{"variable": name, **values} for name, values in stats.items()]
    with pd.option_context("display.max_rows", 200, "display.max_columns", 16, "display.max_colwidth", 80):
        display(pd.DataFrame(rows))


SAMPLE DIAGNOSTICS: /dataset/axis0


,variable,sampled_values,dtype,unique_in_sample,most_common_in_sample
0,value,66,|S16,66,"[(MCFM_dxs_HZZ, 1), (MCFM_dxs_ggZZbox, 1), (MCFM_dxs_ggZZtot, 1), (MCFM_dxs_..."



SAMPLE DIAGNOSTICS: /dataset/axis1


,variable,sampled_values,dtype,nonfinite,nan,positive_infinity,negative_infinity,min,max,mean,std,negative,zero,positive
0,value,2000,int64,0,0,0,0,0,9540479,4770239.5,3.372928e+06,0,1,1999



SAMPLE DIAGNOSTICS: /dataset/block0_items


...

,variable,sampled_values,dtype,unique_in_sample,most_common_in_sample
0,value,7,|S5,7,"[(lepM1, 1), (lepM2, 1), (lepM3, 1), (lepM4, 1), (2jet, 1), (1jet, 1), (labe..."



SAMPLE DIAGNOSTICS: /dataset/block3_values


,variable,sampled_values,dtype,nonfinite,nan,positive_infinity,negative_infinity,min,max,mean,std,negative,zero,positive,sample_value_counts
0,lepM1,2000,int64,0,0,0,0,1,2,1.5465,0.497833,0,0,2000,"[{'value': 1, 'count': 907}, {'value': 2, 'count': 1093}]"
1,lepM2,2000,int64,0,0,0,0,1,2,1.5465,0.497833,0,0,2000,"[{'value': 1, 'count': 907}, {'value': 2, 'count': 1093}]"
2,lepM3,2000,int64,0,0,0,0,1,2,1.5560,0.496854,0,0,2000,"[{'value': 1, 'count': 888}, {'value': 2, 'count': 1112}]"
3,lepM4,2000,int64,0,0,0,0,1,2,1.5560,0.496854,0,0,2000,"[{'value': 1, 'count': 888}, {'value': 2, 'count': 1112}]"
4,2jet,2000,int64,0,0,0,0,0,1,0.3130,0.463714,0,1374,626,"[{'value': 0, 'count': 1374}, {'value': 1, 'count': 626}]"
5,1jet,2000,int64,0,0,0,0,0,1,0.6610,0.473370,0,678,1322,"[{'value': 0, 'count': 678}, {'value': 1, 'count': 1322}]"
6,label,2000,int64,0,0,0,0,0,9,5.0000,4.098780,0,800,1200,"[{'value': 0, 'count': 800}, {'value': 8, 'count': 800}, {'value': 9, 'count..."



SAMPLE DIAGNOSTICS: /dataset/block4_items


,variable,sampled_values,dtype,unique_in_sample,most_common_in_sample
0,value,2,|S7,2,"[(Type, 1), (subType, 1)]"


## 6. Save the reports to Drive

These small report files can be read independently of the 3.6 GB ntuple. Re-running replaces only the generated reports, never the HDF5 input. The full schema and all collected diagnostics remain in the reports even if notebook tables are visually truncated.

In [7]:
text_path = FILE_PATH.with_name(FILE_PATH.stem + ".inspection.txt")
json_path = FILE_PATH.with_name(FILE_PATH.stem + ".inspection.json")

text_path.write_text(make_text_report(report), encoding="utf-8")
json_path.write_text(json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")

for output in (text_path, json_path):
    print(f"Saved: {output} ({output.stat().st_size:,} bytes)")
print("\nDone. Tell me the inspection reports are ready in Drive.")

Saved: /content/drive/MyDrive/Colab Notebooks/HZZ_OffShell/dataset_wDR_mean_origIndex.inspection.txt (46,803 bytes)
Saved: /content/drive/MyDrive/Colab Notebooks/HZZ_OffShell/dataset_wDR_mean_origIndex.inspection.json (56,003 bytes)

Done. Tell me the inspection reports are ready in Drive.
